# Sola Face LoRA (SDXL) — Colab

1. **Runtime → GPU**
2. Upload **`sola_face_kohya.zip`** from local: `foocus_new/datasets/sola_face_kohya.zip`
   (must contain folder `10_sola_face/` with `.jpg` + `.txt`)
3. Run all
4. Download `sola_face_sdxl.safetensors`

Trigger: `sola_face`


In [ ]:
# @title 1) Setup Kohya sd-scripts
import os
os.chdir("/content")
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip -q install xformers --index-url https://download.pytorch.org/whl/cu121 || true
if not os.path.isdir("/content/sd-scripts"):
    !git clone --depth 1 https://github.com/kohya-ss/sd-scripts /content/sd-scripts
os.chdir("/content/sd-scripts")
!pip -q install -r requirements.txt
!pip -q install bitsandbytes
print("OK setup")


In [ ]:
# @title 2) Upload sola_face_kohya.zip + normalize structure
import os, zipfile, shutil, glob
from google.colab import files

DATA = "/content/sola_data"
shutil.rmtree(DATA, ignore_errors=True)
os.makedirs(DATA, exist_ok=True)
os.chdir(DATA)

print("Upload foocus_new/datasets/sola_face_kohya.zip")
uploaded = files.upload()
assert uploaded, "No file uploaded"

for name in uploaded:
    path = os.path.join(DATA, name)
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(DATA)
        print("Extracted", name, "entries", len(z.namelist()))

def list_tree(base, max_level=3):
    for root, dirs, fs in os.walk(base):
        level = root.replace(base, "").count(os.sep)
        if level > max_level:
            continue
        print("  " * level + os.path.basename(root) + "/")
        if level >= 2:
            continue
        for f in sorted(fs)[:12]:
            print("  " * (level + 1) + f)

print("=== after extract ===")
list_tree(DATA)

# Find existing Kohya class folder OR rebuild from flat jpg+txt
found = None
for root, dirs, files in os.walk(DATA):
    for d in dirs:
        if d == "10_sola_face" or (d.startswith("10_") and "sola" in d.lower()):
            found = os.path.join(root, d)
            break
    if found:
        break

if found is None:
    # flat images somewhere?
    jpgs = []
    for root, dirs, files in os.walk(DATA):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                jpgs.append(os.path.join(root, f))
    if len(jpgs) < 10:
        raise FileNotFoundError(
            "Zip wrong. Need folder 10_sola_face/ with images. "
            f"Found only {len(jpgs)} images. Re-upload datasets/sola_face_kohya.zip"
        )
    found = os.path.join(DATA, "10_sola_face")
    os.makedirs(found, exist_ok=True)
    for src in jpgs:
        base = os.path.basename(src)
        stem, _ = os.path.splitext(base)
        dst = os.path.join(found, stem + ".jpg")
        shutil.copy2(src, dst)
        # caption
        txt_src = os.path.splitext(src)[0] + ".txt"
        txt_dst = os.path.join(found, stem + ".txt")
        if os.path.isfile(txt_src):
            shutil.copy2(txt_src, txt_dst)
        else:
            open(txt_dst, "w", encoding="utf-8").write(
                "sola_face, photo of a woman, adult woman, long straight brown hair, middle part, looking at camera\n"
            )
    print("Rebuilt", found, "from flat files", len(jpgs))

# TRAIN_ROOT must be PARENT of 10_sola_face
TRAIN_ROOT = os.path.dirname(found)
CLASS_DIR = found
OUT = "/content/outputs/sola_face_lora"
os.makedirs(OUT, exist_ok=True)

n_img = len([f for f in os.listdir(CLASS_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
n_txt = len([f for f in os.listdir(CLASS_DIR) if f.lower().endswith(".txt")])
print("CLASS_DIR =", CLASS_DIR)
print("TRAIN_ROOT =", TRAIN_ROOT)
print("images =", n_img, "captions =", n_txt)
assert n_img >= 10, "Too few images"


In [ ]:
# @title 3) Train SDXL LoRA (face)
import os
os.chdir("/content/sd-scripts")

cmd = f"""accelerate launch --num_cpu_threads_per_process 2 sdxl_train_network.py \
  --pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 \
  --train_data_dir={TRAIN_ROOT} \
  --output_dir={OUT} \
  --output_name=sola_face_sdxl \
  --save_model_as=safetensors \
  --caption_extension=.txt \
  --resolution=1024,1024 \
  --enable_bucket \
  --min_bucket_reso=512 \
  --max_bucket_reso=2048 \
  --train_batch_size=1 \
  --max_train_epochs=10 \
  --save_every_n_epochs=2 \
  --learning_rate=1e-4 \
  --unet_lr=1e-4 \
  --text_encoder_lr=5e-5 \
  --lr_scheduler=cosine_with_restarts \
  --lr_scheduler_num_cycles=3 \
  --optimizer_type=AdamW8bit \
  --network_module=networks.lora \
  --network_dim=16 \
  --network_alpha=8 \
  --mixed_precision=fp16 \
  --xformers \
  --cache_latents \
  --cache_latents_to_disk \
  --seed=42 \
  --keep_tokens=1 \
  --noise_offset=0.0357 \
  --min_snr_gamma=5 \
  --max_data_loader_n_workers=2
"""
print(cmd)
get_ipython().system(cmd)


In [ ]:
# @title 4) Download LoRA
import glob
from google.colab import files

paths = sorted(glob.glob("/content/outputs/sola_face_lora/*.safetensors"))
print("Found:", paths)
assert paths, "No LoRA produced — check train logs"
final = [p for p in paths if p.endswith("sola_face_sdxl.safetensors")] or [paths[-1]]
for p in final:
    print("Downloading", p)
    files.download(p)
